## A quick note:
This notebook is built to use AI functions on staged images. For this code to execute and return meaningful results, you will need to store an image for AI_CLASSIFY to classify against.

In [ ]:
%%sql -r setup_result
USE DATABASE SKI_GEAR_SUPPORT_DB;
USE SCHEMA SKI_GEAR_SUPPORT_SCHEMA;

CREATE STAGE IF NOT EXISTS returns_stage;

CREATE TABLE IF NOT EXISTS product_returns (
    return_id INT,
    return_photo_path VARCHAR,
    product VARCHAR,
    customer_name VARCHAR,
    return_date DATE
);

In [ ]:
%%sql -r classify_results
SELECT return_id, return_photo_path,
    AI_CLASSIFY(
        TO_FILE('@returns_stage', return_photo_path),
        ['torn_fabric', 'broken_buckle', 'scuffed_surface', 'missing_component', 'no_visible_damage']
    ):labels[0]::STRING AS damage_type
FROM product_returns
WHERE return_photo_path IS NOT NULL;

In [ ]:
SELECT
    product,
    damage_type,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', transcript) AS transcript_embedding
FROM call_transcripts
LIMIT 5;


In [ ]:
-- Setup: Create stage and table for warranty form parsing demo
CREATE STAGE IF NOT EXISTS warranty_forms_stage;
CREATE TABLE IF NOT EXISTS warranty_submissions (
    doc_id INT,
    form_image_path VARCHAR
);

-- Parse warranty form documents with AI_PARSE_DOCUMENT
SELECT
    doc_id,
    AI_PARSE_DOCUMENT(
        TO_FILE('@warranty_forms_stage', form_image_path),
        {'mode': 'LAYOUT'}
    ) AS parsed_content
FROM warranty_submissions;
